# Blinkit Grocery Sales Analysis

This notebook performs reproducible data cleaning, validation and exploratory analysis using the supplied Blinkit-style dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
orders = pd.read_csv(RAW/'blinkit_orders.csv', parse_dates=['order_date'])
items = pd.read_csv(RAW/'blinkit_order_items.csv')
products = pd.read_csv(RAW/'blinkit_products.csv')
customers = pd.read_csv(RAW/'blinkit_customers_anonymized.csv')
delivery = pd.read_csv(RAW/'blinkit_delivery_performance.csv', parse_dates=['promised_time','actual_time'])
marketing = pd.read_csv(RAW/'blinkit_marketing_performance.csv')
inventory = pd.read_csv(RAW/'blinkit_inventory.csv')
feedback = pd.read_csv(RAW/'blinkit_customer_feedback.csv')
print(orders.shape, items.shape, products.shape, customers.shape)

## Data quality checks

In [ ]:
print('Missing values in orders:')
print(orders.isna().sum().sort_values(ascending=False).head(10))
print('Duplicate order IDs:', orders.order_id.duplicated().sum())
print('Product link coverage:', items.product_id.isin(products.product_id).mean())

## Derived fields

In [ ]:
delivery['delivery_variance_minutes'] = (delivery.actual_time - delivery.promised_time).dt.total_seconds()/60
items = items.merge(products[['product_id','product_name','category','margin_percentage']], on='product_id', how='left')
items['line_sales'] = items.quantity * items.unit_price
items['estimated_gross_margin'] = items.line_sales * items.margin_percentage / 100
print(items[['line_sales','estimated_gross_margin']].describe())

## Sales and customer analysis

In [ ]:
print('Total order revenue:', round(orders.order_total.sum(),2))
print('Average order value:', round(orders.order_total.mean(),2))
repeat_rate = orders.groupby('customer_id').size().gt(1).mean()
print('Repeat customer rate:', round(repeat_rate*100,2),'%')

## Product analysis

In [ ]:
cat = items.groupby('category').line_sales.sum().sort_values(ascending=False)
print(cat)
cat.plot(kind='bar', title='Item-level Sales by Category')
plt.ylabel('Sales (₹)')
plt.tight_layout()
plt.show()

## Delivery analysis

In [ ]:
print(orders.delivery_status.value_counts(normalize=True).mul(100).round(2))
print('Average timestamp variance:', round(delivery.delivery_variance_minutes.mean(),2),'minutes')

## Marketing and inventory

In [ ]:
m = pd.read_csv(RAW/'blinkit_marketing_performance.csv')
channel = m.groupby('channel').agg(spend=('spend','sum'), revenue=('revenue_generated','sum'), clicks=('clicks','sum'), impressions=('impressions','sum'))
channel['roas'] = channel.revenue/channel.spend
channel['ctr'] = channel.clicks/channel.impressions
print(channel.sort_values('roas', ascending=False))
inv = pd.read_csv(RAW/'blinkit_inventory.csv')
print('Inventory damage rate:', round(inv.damaged_stock.sum()/inv.stock_received.sum()*100,2),'%')

## Interview takeaway

The project deliberately separates order-level revenue from item-level sales because the source data does not reconcile them. It also avoids claiming store-level performance because store IDs are unique per order.